In [0]:
-- Stored procedures
/* A stored procedure in SQL is a named, precompiled group of SQL statements saved directly inside a database. You can call and reuse it repeatedly like a function in a programming language. */

-- Total number of US customer and total of score
SELECT
    count(*) AS totalCustomer,
    round(avg(score), 2) AS averageScore
FROM workspace.default.customers
WHERE country = 'USA';

-- CREATE stored procedure
CREATE OR REPLACE PROCEDURE workspace.default.getCustomerSummary()
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN
    SELECT
        count(*) AS totalCustomer,
        round(avg(score), 2) AS averageScore
    FROM workspace.default.customers
    WHERE country = 'USA';
END;


-- run stored procedure
CALL workspace.default.getCustomerSummary();

-- parameterized procedure
CREATE OR REPLACE PROCEDURE workspace.default.getCustomerSummary(IN country STRING)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN
    SELECT
        count(*) AS totalCustomer,
        round(avg(score), 2) AS averageScore
    FROM workspace.default.customers
    WHERE country = country;
END;

-- call stored procedure
-- Positional argument
CALL workspace.default.getCustomerSummary('USA');

-- Or named argument
CALL workspace.default.getCustomerSummary(country => 'Germany');


-- multiple statements in SP
CREATE OR REPLACE PROCEDURE workspace.default.getCustomerSummary(
    IN country STRING,
    OUT totalCustomer INT,
    OUT averageScore FLOAT,
    OUT totalProduct INT,
    OUT totalSales FLOAT)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN
    SELECT
        count(*) AS totalCustomer,
        round(avg(score), 2) AS averageScore
    FROM workspace.default.customers
    WHERE country = country;

    -- total product and total sales
    SELECT
        count(o.cust_id) AS totalProduct,
        SUM(sales) toalaSales
    FROM workspace.default.orders o
    JOIN workspace.default.customers c
    ON c.cust_id = o.cust_id
    WHERE country = country;
END;

-- session variables to receive the OUT parameter values
DECLARE VARIABLE totalCustomers INT;
DECLARE VARIABLE averageScores FLOAT;
DECLARE VARIABLE totalProducts INT;
DECLARE VARIABLE totalSales FLOAT;

-- call supplying ALL parameters: country positionally, OUT params as variables
CALL workspace.default.getCustomerSummary(
    'Germany',
    totalCustomers,
    averageScores,
    totalProducts,
    totalSales);

-- read back the OUT values
SELECT
    totalCustomers,
    averageScores,
    totalProducts,
    totalSales;

-- variable in Stored Procedure
CREATE OR REPLACE PROCEDURE workspace.default.getCustomerSummary(IN p_country STRING)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN
    -- 1. Declare local variables
    DECLARE totalCustomer BIGINT;
    DECLARE averageScore DOUBLE;

    -- 2. Store SELECT results into the variables
    SET (totalCustomer, averageScore) = (
        SELECT
            count(*),
            round(avg(score), 2)
        FROM workspace.default.customers
        WHERE country = p_country
    );

    -- 3. Output the formatted messages as a result set
    SELECT 
        CONCAT('Total customers from ', p_country, ' is ', CAST(totalCustomer AS STRING)) AS customer_message,
        CONCAT('Average score is ', CAST(averageScore AS STRING)) AS score_message;
END;

CALL workspace.default.getCustomerSummary(p_country => 'Germany');

























